# Correlation Analysis (Table 1 + Figure 1)

Reproduces **Table 1** and **Figure 1** from the DTR paper.

- **Table 1**: Pearson correlation between 7 thinking metrics and task accuracy across model-benchmark pairs
- **Figure 1**: Scatter plots comparing DTR vs accuracy (top row) and token count vs accuracy (bottom row)

The key finding is that DTR achieves the highest and most consistent correlation with accuracy,
outperforming simpler metrics like token count.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

from dtr.metrics.dtr import compute_dtr
from dtr.metrics.baselines import compute_all_baselines
from dtr.analysis.correlation import (
    compute_binned_correlation,
    compute_correlation_table,
    average_over_seeds,
)
from dtr.utils.io import load_json, load_jsonl

## Configuration

In [ ]:
MODELS = ["deepseek-r1", "qwq-32b"]
BENCHMARKS = ["aime_2025", "hmmt_2025", "gpqa_diamond", "livecodebench"]
METRICS_DIR = Path("..") / "results" / "metrics"

METRIC_NAMES = [
    "DTR",
    "Token Count",
    "Reverse Token Count",
    "Log Prob",
    "Neg Perplexity",
    "Neg Entropy",
    "Self-Certainty",
]

METRIC_KEYS = [
    "dtr",
    "token_count",
    "reverse_token_count",
    "log_prob",
    "neg_perplexity",
    "neg_entropy",
    "self_certainty",
]

## Load Pre-Computed Metrics

Load per-sample metrics for all model-benchmark combinations.

In [ ]:
all_results = {}

for model in MODELS:
    for bench in BENCHMARKS:
        key = f"{model}/{bench}"
        metrics_path = METRICS_DIR / model / bench / "per_sample_metrics.json"
        
        if metrics_path.exists():
            data = load_json(str(metrics_path))
            all_results[key] = data
            print(f"Loaded {key}: {len(data)} samples")
        else:
            print(f"WARNING: {metrics_path} not found")

print(f"\nLoaded {len(all_results)} model-benchmark combinations")

## Table 1: Correlation Table

Pearson r between each metric and accuracy, computed using binned means.
Each cell shows the correlation for a specific (metric, model-benchmark) pair.
The DTR row and Token Count row are highlighted for comparison.

In [ ]:
# Build correlation table
corr_table = {}

for key, data in all_results.items():
    accuracies = np.array([d["correct"] for d in data], dtype=float)
    
    col_corrs = {}
    for metric_name, metric_key in zip(METRIC_NAMES, METRIC_KEYS):
        if all(metric_key in d for d in data):
            metric_values = np.array([d[metric_key] for d in data])
            binned_corr = compute_binned_correlation(metric_values, accuracies)
            col_corrs[metric_name] = binned_corr
        else:
            col_corrs[metric_name] = np.nan
    
    corr_table[key] = col_corrs

# Create DataFrame
table1_df = pd.DataFrame(corr_table, index=METRIC_NAMES)
table1_df.columns = [c.replace("/", "\n") for c in table1_df.columns]

print("Table 1: Pearson r (Binned Correlation) Between Metrics and Accuracy")
print("=" * 80)
table1_df

In [ ]:
# Styled version with highlights
def highlight_rows(row):
    if row.name == "DTR":
        return ["background-color: #c6efce; font-weight: bold"] * len(row)
    elif row.name == "Token Count":
        return ["background-color: #fce4d6"] * len(row)
    return [""] * len(row)

styled_table = (
    table1_df.style
    .apply(highlight_rows, axis=1)
    .format("{:.3f}")
    .set_caption("Table 1: Binned Pearson r between thinking metrics and accuracy")
)
styled_table

## Figure 1: DTR vs Token Count Scatter Plots

2x4 grid of scatter plots:
- **Top row**: DTR vs accuracy for each benchmark
- **Bottom row**: Token count vs accuracy for each benchmark

Points are binned and shown with error bars to reveal the underlying trend.

In [ ]:
# Use the first model for Figure 1 (or combine if both available)
PLOT_MODEL = MODELS[0]
NUM_BINS = 10

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle(
    f"Figure 1: DTR vs Token Count Correlation with Accuracy ({PLOT_MODEL})",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)

colors = {"DTR": sns.color_palette()[0], "Token Count": sns.color_palette()[1]}

for col_idx, bench in enumerate(BENCHMARKS):
    key = f"{PLOT_MODEL}/{bench}"
    if key not in all_results:
        for ax_row in range(2):
            axes[ax_row, col_idx].text(0.5, 0.5, "No data", ha="center", va="center",
                                        transform=axes[ax_row, col_idx].transAxes)
        continue
    
    data = all_results[key]
    accuracies = np.array([d["correct"] for d in data], dtype=float)
    
    for row_idx, (metric_name, metric_key) in enumerate([("DTR", "dtr"), ("Token Count", "token_count")]):
        ax = axes[row_idx, col_idx]
        
        metric_values = np.array([d.get(metric_key, 0) for d in data])
        
        # Create bins
        bin_edges = np.percentile(metric_values, np.linspace(0, 100, NUM_BINS + 1))
        bin_edges = np.unique(bin_edges)
        bin_centers = []
        bin_means = []
        bin_stds = []
        
        for i in range(len(bin_edges) - 1):
            mask = (metric_values >= bin_edges[i]) & (metric_values < bin_edges[i + 1])
            if i == len(bin_edges) - 2:
                mask = (metric_values >= bin_edges[i]) & (metric_values <= bin_edges[i + 1])
            if mask.sum() > 0:
                bin_centers.append(metric_values[mask].mean())
                bin_means.append(accuracies[mask].mean())
                bin_stds.append(accuracies[mask].std() / np.sqrt(mask.sum()))
        
        # Scatter (individual points, faded)
        ax.scatter(
            metric_values, accuracies,
            alpha=0.1, s=8, color=colors[metric_name], rasterized=True,
        )
        
        # Binned means with error bars
        ax.errorbar(
            bin_centers, bin_means, yerr=bin_stds,
            fmt="o-", color=colors[metric_name], markersize=6,
            linewidth=2, capsize=3, label=f"Binned mean",
        )
        
        # Compute correlation
        r, p = stats.pearsonr(bin_centers, bin_means) if len(bin_centers) > 2 else (np.nan, np.nan)
        ax.text(
            0.05, 0.95, f"r = {r:.3f}",
            transform=ax.transAxes, fontsize=11, fontweight="bold",
            verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
        )
        
        ax.set_ylabel("Accuracy" if col_idx == 0 else "")
        if row_idx == 0:
            ax.set_title(bench.replace("_", " ").title(), fontsize=12)
        if row_idx == 1:
            ax.set_xlabel(metric_name)
        else:
            ax.set_xlabel(metric_name)
        
        ax.legend(fontsize=8, loc="lower right")

# Row labels
axes[0, 0].annotate(
    "DTR", xy=(0, 0.5), xytext=(-axes[0, 0].yaxis.labelpad - 15, 0),
    xycoords=axes[0, 0].yaxis.label, textcoords="offset points",
    fontsize=14, fontweight="bold", ha="right", va="center", rotation=90,
)
axes[1, 0].annotate(
    "Token\nCount", xy=(0, 0.5), xytext=(-axes[1, 0].yaxis.labelpad - 15, 0),
    xycoords=axes[1, 0].yaxis.label, textcoords="offset points",
    fontsize=14, fontweight="bold", ha="right", va="center", rotation=90,
)

plt.tight_layout()
plt.savefig("../results/figures/fig1_correlation_scatter.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Average Correlation Across Seeds

If multiple seeds are available, compute the mean and standard deviation of correlations.

In [ ]:
# Check for multi-seed results
seed_dirs = sorted(METRICS_DIR.glob("*/*/seed_*"))
if seed_dirs:
    print(f"Found {len(seed_dirs)} seed directories")
    # average_over_seeds would aggregate these
    # avg_table = average_over_seeds(METRICS_DIR, MODELS, BENCHMARKS)
    # display(avg_table)
else:
    print("No multi-seed results found. Table 1 uses single-seed correlations.")